In [7]:
import tempfile
import os

import sklearn
from sklearn.datasets import load_diabetes
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn


def main():
    # Setup MLflow
    mlflow.set_experiment("elasticnet-diabetes")
    
    # Load data
    diabetes = load_diabetes()
    X_train, X_test, y_train, y_test = train_test_split(
        diabetes.data, diabetes.target, test_size=0.2, random_state=42
    )
    
    # Train model
    model = ElasticNet(alpha=0.5, l1_ratio=0.5)
    model.fit(X_train, y_train)
    
    # Get sklearn version
    sklearn_req = f"scikit-learn=={sklearn.__version__}"
    numpy_req = "numpy>=1.20.0"
    
    with mlflow.start_run():
        # Log metrics
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
        mlflow.log_metric("train_r2", train_score)
        mlflow.log_metric("test_r2", test_score)
        
        print(f"Train R²: {train_score:.3f}")
        print(f"Test R²: {test_score:.3f}")
        
        # Method 1: Default requirements (auto-detected)
        print("\n1. Logging with default requirements...")
        mlflow.sklearn.log_model(model, "model_default")
        
        # Method 2: Custom requirements list
        print("2. Logging with custom requirements...")
        mlflow.sklearn.log_model(
            model, 
            "model_custom",
            pip_requirements=[sklearn_req, numpy_req]
        )
        
        # Method 3: Extra requirements (adds to defaults)
        print("3. Logging with extra requirements...")
        mlflow.sklearn.log_model(
            model,
            "model_extra",
            extra_pip_requirements=["pandas>=1.3.0"]
        )
        
        # Method 4: Requirements from file
        print("4. Logging with requirements file...")
        with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
            f.write(f"{sklearn_req}\n")
            f.write(f"{numpy_req}\n")
            f.write("pandas>=1.3.0\n")
            temp_path = f.name
        
        try:
            mlflow.sklearn.log_model(
                model,
                "model_from_file",
                pip_requirements=temp_path
            )
        finally:
            os.unlink(temp_path)
        
        print("\n✓ All models logged successfully!")
        print(f"Run ID: {mlflow.active_run().info.run_id}")


if __name__ == "__main__":
    main()

2025/10/31 13:09:56 INFO mlflow.tracking.fluent: Experiment with name 'elasticnet-diabetes' does not exist. Creating a new experiment.
2025/10/31 13:09:56 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'e5bbf9f2472f4bcc8e081501661d5d5b', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/10/31 13:09:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Train R²: 0.021
Test R²: 0.011

1. Logging with default requirements...


2025/10/31 13:10:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/31 13:10:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/31 13:10:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/31 13:10:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2. Logging with custom requirements...
3. Logging with extra requirements...


2025/10/31 13:10:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/10/31 13:10:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/31 13:10:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


4. Logging with requirements file...

✓ All models logged successfully!
Run ID: 13b0e11741f042d4a809dcd5afaeb1dc


In [11]:
import subprocess
import time

ui_process = subprocess.Popen(
    ["mlflow", "ui", "--port", "5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)  # Give it time to start
print("✓ MLflow UI started at http://127.0.0.1:5000")
print("Check your browser!")

✓ MLflow UI started at http://127.0.0.1:5000
Check your browser!


In [13]:
ui_process.terminate()
print("✓ MLflow UI stopped")

✓ MLflow UI stopped
